# PPO (Proximal Policy Optimization) Clipped Loss

**难度:** Medium

实现 PPO 裁剪代理损失。

PPO 通过裁剪重要性采样比率来约束策略更新，防止强化学习中的破坏性大幅更新。

**签名:** `ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2) -> Tensor`

**参数:**
- `new_logps` — 当前策略对数概率 (B,)
- `old_logps` — 旧策略对数概率 (B,)，视为常量
- `advantages` — 优势估计 (B,)，视为常量
- `clip_ratio` — 裁剪范围 epsilon

**返回:** 标量损失

**约束:**
- 比率：`r = exp(new_logps - old_logps.detach())`
- 损失：`-min(r * A, clamp(r, 1-eps, 1+eps) * A).mean()`
- 梯度仅通过 new_logps 流动

**提示:**
1. r = exp(new_logps - old_logps.detach())
2. unclipped = r * advantages.detach()
3. clipped = clamp(r, 1-ε, 1+ε) * advantages.detach()
4. loss = -min(unclipped, clipped).mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 PPO Clipped Loss ----
# 面试高频题：PPO 的裁剪机制是 RLHF 的核心

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    # detach 确保梯度只通过 new_logps 流动
    old_logps_detached = old_logps.detach()
    adv_detached = advantages.detach()

    # 面试考点：为什么用 exp(log_ratio) 而不是直接除？
    # 答：log 空间做减法比概率空间做除法更数值稳定
    # 当概率很小时，除法可能得到 inf 或极大值
    ratios = torch.exp(new_logps - old_logps_detached)

    unclipped = ratios * adv_detached
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_detached

    # 面试高频考点：为什么取 min 而不是 max？
    # 画图理解：
    # A > 0 时：unclipped 随 r 增大而增大，clipped 在 r>1+ε 后不变
    #           min 取 clipped → 限制了上界（不能过度增加好 action 的概率）
    # A < 0 时：unclipped 随 r 增大而减小（更负），clipped 在 r>1+ε 后不变
    #           min 取 unclipped → 限制了下界（不能过度降低差 action 的概率）
    # 总结：min 确保两个方向都被 clip，这就是"proximal"的含义
    return -torch.min(unclipped, clipped).mean()

In [ ]:
# Demo
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


In [ ]:
from torch_judge import check
check('ppo_loss')


## 📝 核心思路总结

1. **PPO 的核心：裁剪约束策略更新幅度**：防止新策略偏离旧策略太远导致训练崩溃
2. **重要性采样比率 r = π_new/π_old**：用旧数据估计新策略的梯度（off-policy）
3. **min 取保守方向**：当 A>0 时 clip 上界，A<0 时 clip 下界，确保更新幅度有界